In [10]:
# Retry with a simpler (and more robust) symmetry test:
# Enforce R^T A R = A at three distinct angles and solve for (a,b,c).
# Also include the infinitesimal-rotation commutator proof.

import sympy as sp

a, b, c = sp.symbols('a b c', real=True)
theta = sp.symbols('theta', real=True)

R = sp.Matrix([[sp.cos(theta), -sp.sin(theta)],
               [sp.sin(theta),  sp.cos(theta)]])
A = sp.Matrix([[a, b],
               [b, c]])

RtAR = sp.simplify(R.T * A * R)
Delta = sp.simplify(RtAR - A)

# Build equations by sampling Δ(θ) at a small set of angles (enough to kill sin/cos components)
angles = [0, sp.pi/4, sp.pi/2]
eqs = []
for ang in angles:
    Dv = sp.simplify(Delta.subs(theta, ang))
    # require every entry to be zero at this angle
    for i in range(2):
        for j in range(2):
            eqs.append(sp.Eq(sp.simplify(Dv[i,j]), 0))

sol = sp.solve(eqs, (a,b,c), dict=True)

print("=== Solving invariance from R^T A R = A at θ ∈ {0, π/4, π/2} ===")
print("Equations:")
for e in eqs:
    print(" ", e)
print("\nSolutions for (a,b,c):", sol)

# Derive explicit relations rather than a concrete triple.
# Typically SymPy gives a relation set; if it returns an empty list (meaning underdetermined),
# we can compute relations from a single symbolic off-diagonal entry.
if not sol:
    off = sp.simplify(Delta[0,1])
    off2 = sp.simplify(sp.expand_trig(off.rewrite(sp.sin)))
    print("\nOff-diagonal Δ12(θ):", off2)
    # Express off2 = A*sin(2θ) + B*cos(2θ). Extract A and B by evaluating at θ=0 and θ=π/4.
    Acoef = sp.simplify(sp.diff(off2, sp.sin(2*theta)).subs({theta: sp.pi/4}))
    Bcoef = sp.simplify(sp.diff(off2, sp.cos(2*theta)).subs({theta: 0}))
    print("Heuristic coefficients A (sin 2θ) and B (cos 2θ):", Acoef, Bcoef)

# Infinitesimal method
J = sp.Matrix([[0, -1],
               [1,  0]])
comm = sp.simplify(A*J - J*A)
eqs_comm = [sp.Eq(comm[0,0],0), sp.Eq(comm[0,1],0), sp.Eq(comm[1,0],0), sp.Eq(comm[1,1],0)]
sol_comm = sp.solve(eqs_comm, (a,b,c), dict=True)

print("\n=== Infinitesimal method ([A,J]=0) ===")
print("Commutator [A,J]:")
sp.pprint(comm)
print("\nSolutions for (a,b,c) from [A,J]=0:", sol_comm)

# As a final sanity check, verify that A = α I satisfies invariance symbolically.
alpha = sp.symbols('alpha', real=True)
A_iso = alpha * sp.eye(2)
check = sp.simplify(R.T * A_iso * R - A_iso)
print("\nSanity check: with A = α I, R^T A R - A equals")
sp.pprint(check)

=== Solving invariance from R^T A R = A at θ ∈ {0, π/4, π/2} ===
Equations:
  True
  True
  True
  True
  Eq(-a/2 + b + c/2, 0)
  Eq(-a/2 - b + c/2, 0)
  Eq(-a/2 - b + c/2, 0)
  Eq(a/2 - b - c/2, 0)
  Eq(-a + c, 0)
  Eq(-2*b, 0)
  Eq(-2*b, 0)
  Eq(a - c, 0)

Solutions for (a,b,c): [{a: c, b: 0}]

=== Infinitesimal method ([A,J]=0) ===
Commutator [A,J]:
⎡ 2⋅b    -a + c⎤
⎢              ⎥
⎣-a + c   -2⋅b ⎦

Solutions for (a,b,c) from [A,J]=0: [{a: c, b: 0}]

Sanity check: with A = α I, R^T A R - A equals
⎡0  0⎤
⎢    ⎥
⎣0  0⎦


# 3

In [11]:
# SymPy check: normalize, complete the square, extract center & radius, and classify regimes symbolically.

import sympy as sp

# Symbols
x, y = sp.symbols('x y', real=True)
a, d, e, f = sp.symbols('a d e f', real=True, nonzero=True)

# Assume isotropy: b=0, a=c != 0; start from a x^2 + a y^2 + 2 d x + 2 e y + f = 0
expr = a*x**2 + a*y**2 + 2*d*x + 2*e*y + f

# Normalize by a and rename
m, n, q = sp.symbols('m n q', real=True)
subs_map = {m: d/a, n: e/a, q: f/a}
expr_norm = sp.simplify((expr / a).xreplace({d/a: m, e/a: n, f/a: q}))  # xreplace to keep m,n,q explicitly

# Complete the square: (x+m)^2 + (y+n)^2 - (m^2 + n^2 - q) == 0
completed = sp.expand((x + m)**2 + (y + n)**2 - (m**2 + n**2 - q))

# Extract center and radius^2
center = (-m, -n)
R2 = sp.simplify(m**2 + n**2 - q)

print("Normalized equation:")
sp.pprint(sp.Eq(expr_norm, 0))

print("\nCompleted-square form:")
sp.pprint(sp.Eq(completed, 0))

print("\nCenter C and squared radius R^2:")
print("C =", center)
print("R^2 =", R2)

# Regime classification (symbolic description)
print("\nRegimes:")
print("1) R^2 > 0  → circle of radius sqrt(R^2).")
print("2) R^2 = 0  → point circle at C.")
print("3) R^2 < 0  → empty locus (no real points).")

Normalized equation:
   2      2                        
a⋅x  + a⋅y  + 2⋅d⋅x + 2⋅e⋅y + f    
─────────────────────────────── = 0
               a                   

Completed-square form:
                     2    2    
2⋅m⋅x + 2⋅n⋅y + q + x  + y  = 0

Center C and squared radius R^2:
C = (-m, -n)
R^2 = m**2 + n**2 - q

Regimes:
1) R^2 > 0  → circle of radius sqrt(R^2).
2) R^2 = 0  → point circle at C.
3) R^2 < 0  → empty locus (no real points).


# 4

In [12]:
# SymPy program for the translation X = Y + h that removes linear terms in X^T A X + 2 ℓ^T X + f = 0.
# - Shows that the linear term vanishes iff A h = -ℓ
# - Solves h symbolically (when det(A) ≠ 0)
# - Specializes to the isotropic case A = λ I to get h = -(1/λ) ℓ
# - Connects to completed-square parameters for a circle: (b=0, a=c=λ) ⇒ h = (-m, -n) with m=d/λ, n=e/λ
# - Provides a helper to compute the translated constant term f' = h^T A h + 2 ℓ^T h + f
# - Includes a quick numeric demo
#
# Plain printed output.

import sympy as sp

# Symbols
a, b, c, d, e, f = sp.symbols('a b c d e f', real=True)
x, y = sp.symbols('x y', real=True)

# Build A, ℓ, X
A = sp.Matrix([[a, b],
               [b, c]])
ell = sp.Matrix([d, e])
X = sp.Matrix([x, y])

# Generic translation vector h
h1, h2 = sp.symbols('h1 h2', real=True)
h = sp.Matrix([h1, h2])
Y = X - h  # (we'll also use X = Y + h equivalently)

# Original quadratic form and translated form
F = (X.T * A * X)[0] + 2*(ell.T * X)[0] + f
F_translated = ((Y + h).T * A * (Y + h))[0] + 2*(ell.T * (Y + h))[0] + f
F_translated_simplified = sp.expand(F_translated)

# Extract the linear-in-Y coefficient vector by taking gradient wrt Y and evaluating at general Y
Y1, Y2 = sp.symbols('Y1 Y2', real=True)
Ysym = sp.Matrix([Y1, Y2])
F_Y = ((Ysym + h).T * A * (Ysym + h))[0] + 2*(ell.T * (Ysym + h))[0] + f
gradY = sp.Matrix([sp.diff(F_Y, Y1), sp.diff(F_Y, Y2)])
linear_coeff = sp.simplify(gradY.subs({Y1:0, Y2:0}))  # coefficient of Y (since gradient at 0)

print("F(X) = X^T A X + 2 ℓ^T X + f")
print("A ="); sp.pprint(A)
print("ℓ ="); sp.pprint(ell)
print("\nTranslated form F(Y+h) expanded:")
sp.pprint(F_translated_simplified)

print("\nLinear-in-Y coefficient (should be 2*(A h + ℓ)):")
sp.pprint(linear_coeff)

print("\nCondition to remove linear terms: A h + ℓ = 0  ⇔  A h = -ℓ")
Ah_plus_ell = sp.simplify(A*h + ell)
print("A h + ℓ =")
sp.pprint(Ah_plus_ell)

# Solve A h = -ℓ symbolically
print("\nSolve A h = -ℓ (when det(A) ≠ 0):")
detA = sp.simplify(A.det())
print("det(A) =", detA)
h_solution = sp.simplify(A.LUsolve(-ell))  # assumes det(A) != 0
print("h* ="); sp.pprint(h_solution)

# Translated constant term f' = h^T A h + 2 ℓ^T h + f
fprime = sp.simplify((h_solution.T * A * h_solution)[0] + 2*(ell.T * h_solution)[0] + f)
print("\nf' (translated constant) = h^T A h + 2 ℓ^T h + f  evaluated at h*")
sp.pprint(fprime)

# Specialization: isotropic case A = λ I
lam = sp.symbols('lam', real=True, nonzero=True)
A_iso = lam * sp.eye(2)
h_iso = sp.simplify(A_iso.LUsolve(-ell))
print("\nIsotropic case A = λ I ⇒ h = -(1/λ) ℓ:")
print("h_iso ="); sp.pprint(h_iso)

# Connection to completed square for the circle case (b=0, a=c=λ):
# In normalized circle form x^2 + y^2 + 2 m x + 2 n y + q = 0, we have m = d/λ, n = e/λ → center = (-m, -n)
m_sym, n_sym, q_sym = sp.symbols('m_sym n_sym q_sym', real=True)
m_from_coeffs = sp.simplify(d/lam)
n_from_coeffs = sp.simplify(e/lam)
center_from_completed_square = sp.Matrix([-m_from_coeffs, -n_from_coeffs])
print("\nCompleted-square center for circle with a=c=λ, b=0:  (-m, -n) = (-d/λ, -e/λ)")
sp.pprint(center_from_completed_square)
print("Matches h_iso:", sp.simplify(center_from_completed_square - h_iso) == sp.Matrix([0,0]))

# Quick numeric demo
print("\nNumeric demo: a=c=4, b=0, d=6, e=-8, f=9  ⇒  x^T A x + 2 ℓ^T x + f = 0")
demo_subs = {a:4, b:0, c:4, d:6, e:-8, f:9}
h_num = sp.N(h_solution.subs(demo_subs))
detA_num = sp.N(detA.subs(demo_subs))
fprime_num = sp.N(fprime.subs(demo_subs))
print("det(A) =", detA_num)
print("h* =", h_num, "  (this should equal the circle center)")
print("f' =", fprime_num, "  (constant term after translation)")

F(X) = X^T A X + 2 ℓ^T X + f
A =
⎡a  b⎤
⎢    ⎥
⎣b  c⎦
ℓ =
⎡d⎤
⎢ ⎥
⎣e⎦

Translated form F(Y+h) expanded:
   2                2                    
a⋅x  + 2⋅b⋅x⋅y + c⋅y  + 2⋅d⋅x + 2⋅e⋅y + f

Linear-in-Y coefficient (should be 2*(A h + ℓ)):
⎡2⋅a⋅h₁ + 2⋅b⋅h₂ + 2⋅d⎤
⎢                     ⎥
⎣2⋅b⋅h₁ + 2⋅c⋅h₂ + 2⋅e⎦

Condition to remove linear terms: A h + ℓ = 0  ⇔  A h = -ℓ
A h + ℓ =
⎡a⋅h₁ + b⋅h₂ + d⎤
⎢               ⎥
⎣b⋅h₁ + c⋅h₂ + e⎦

Solve A h = -ℓ (when det(A) ≠ 0):
det(A) = a*c - b**2
h* =
⎡b⋅e - c⋅d ⎤
⎢───────── ⎥
⎢       2  ⎥
⎢a⋅c - b   ⎥
⎢          ⎥
⎢-a⋅e + b⋅d⎥
⎢──────────⎥
⎢        2 ⎥
⎣ a⋅c - b  ⎦

f' (translated constant) = h^T A h + 2 ℓ^T h + f  evaluated at h*
           2    2                  2
a⋅c⋅f - a⋅e  - b ⋅f + 2⋅b⋅d⋅e - c⋅d 
────────────────────────────────────
                     2              
              a⋅c - b               

Isotropic case A = λ I ⇒ h = -(1/λ) ℓ:
h_iso =
⎡-d ⎤
⎢───⎥
⎢lam⎥
⎢   ⎥
⎢-e ⎥
⎢───⎥
⎣lam⎦

Completed-square center for circle with a=c=λ,

# 5

In [14]:
import sympy as sp

a,b,c,theta = sp.symbols('a b c theta', real=True)
u = sp.Matrix([sp.cos(theta), sp.sin(theta)])
A = sp.Matrix([[a,b],[b,c]])

Q = (u.T * A * u)[0]                  # u^T A u
dQ = sp.simplify(sp.diff(Q, theta))   # must vanish for all theta
# Enforce coefficient-wise zero by evaluating at enough angles:
sol = sp.solve([
    sp.Eq(dQ.subs(theta, 0), 0),
    sp.Eq(dQ.subs(theta, sp.pi/4), 0),
    sp.Eq(dQ.subs(theta, sp.pi/2), 0),
], (a,b,c), dict=True)

print(sol)

[{a: c, b: 0}]


# 6

In [15]:
# SymPy verification for Exercise 6: degeneracies of x^2 + y^2 + 2mx + 2ny + q = 0
import sympy as sp

# Symbols
x, y, m, n, q = sp.symbols('x y m n q', real=True)

# Original and completed-square forms
expr = x**2 + y**2 + 2*m*x + 2*n*y + q
S = sp.simplify(m**2 + n**2 - q)
completed = sp.expand((x + m)**2 + (y + n)**2 - S)   # equals original LHS

print("Original equation:  x^2 + y^2 + 2mx + 2ny + q = 0")
print("Completed square:   (x + m)^2 + (y + n)^2 = S,  where S := m^2 + n^2 - q\n")
print("SymPy check that LHS equals completed-square form minus S:", sp.simplify(expr - ((x+m)**2 + (y+n)**2 - S)) == 0)

# Case 1: S = 0  → unique point (-m, -n)
sol_point = sp.solve([sp.Eq((x+m)**2 + (y+n)**2, 0)], (x, y), dict=True)
print("\nCase S = 0: Solve (x + m)^2 + (y + n)^2 = 0")
print("Solutions:", sol_point, "  (over the reals, squares sum to 0 iff each is 0).")

# Verify back-substitution gives 0 in the original
x0, y0 = -m, -n
check_point = sp.simplify(expr.subs({x: x0, y: y0, q: m**2 + n**2}))
print("Back-substitution at (-m, -n) with q = m^2 + n^2 →", check_point)

# Case 2: S < 0  → no real solutions
# Analytical certificate: min_{x,y} (x+m)^2 + (y+n)^2 = 0, achieved at (-m,-n); cannot equal negative S.
# We'll illustrate with a concrete example.
m_val, n_val = 1, 2

# Example: S = 0 (point circle)
q_point = m_val**2 + n_val**2
expr_point = expr.subs({m: m_val, n: n_val, q: q_point})
lhs_at_center_point = sp.simplify(expr_point.subs({x: -m_val, y: -n_val}))

# Example: S < 0 (no real locus)
q_imag = m_val**2 + n_val**2 + 1   # so S = -1
expr_imag = expr.subs({m: m_val, n: n_val, q: q_imag})
lhs_at_center_imag = sp.simplify(expr_imag.subs({x: -m_val, y: -n_val}))

print("\nConcrete examples with m=1, n=2:")
print("  Example S=0:   q =", q_point, " → equation: x^2 + y^2 + 2x + 4y + 5 = 0")
print("    LHS at (-1, -2):", lhs_at_center_point, " (should be 0)")

print("  Example S<0:  q =", q_imag, " → equation: x^2 + y^2 + 2x + 4y + 6 = 0")
print("    LHS at (-1, -2):", lhs_at_center_imag, " (minimal possible; still > 0 ⇒ no real solutions)")

# Bonus: demonstrate the minimum of (x+m)^2 + (y+n)^2 over reals is 0
min_val = sp.simplify(sp.minimum((x+m)**2 + (y+n)**2, x), rational=True) if hasattr(sp, "minimum") else 0
print("\nAnalytic fact: min_{x,y∈R} (x+m)^2 + (y+n)^2 = 0 at (x,y)=(-m,-n).")

Original equation:  x^2 + y^2 + 2mx + 2ny + q = 0
Completed square:   (x + m)^2 + (y + n)^2 = S,  where S := m^2 + n^2 - q

SymPy check that LHS equals completed-square form minus S: True

Case S = 0: Solve (x + m)^2 + (y + n)^2 = 0
Solutions: [{x: -m - I*(n + y)}, {x: -m + I*(n + y)}]   (over the reals, squares sum to 0 iff each is 0).
Back-substitution at (-m, -n) with q = m^2 + n^2 → 0

Concrete examples with m=1, n=2:
  Example S=0:   q = 5  → equation: x^2 + y^2 + 2x + 4y + 5 = 0
    LHS at (-1, -2): 0  (should be 0)
  Example S<0:  q = 6  → equation: x^2 + y^2 + 2x + 4y + 6 = 0
    LHS at (-1, -2): 1  (minimal possible; still > 0 ⇒ no real solutions)

Analytic fact: min_{x,y∈R} (x+m)^2 + (y+n)^2 = 0 at (x,y)=(-m,-n).


# 7

In [16]:
# SymPy verification for Exercise 7: circle ∩ line, discriminant, and tangency = distance condition
import sympy as sp

# Symbols
x, y = sp.symbols('x y', real=True)
x0, y0, R = sp.symbols('x0 y0 R', real=True)
p, r = sp.symbols('p r', real=True)

# Circle and line
circle = (x - x0)**2 + (y - y0)**2 - R**2
line = sp.Eq(y, p*x + r)

# Substitute y from the line into the circle to get a quadratic in x
expr_x = sp.expand(circle.subs(y, p*x + r))

# Identify quadratic coefficients A x^2 + B x + C
A = sp.expand(sp.diff(expr_x, x, 2) / 2)
B = sp.expand(sp.diff(expr_x, x) - 2*A*x)  # remove the 2Ax part from the first derivative
C = sp.expand(expr_x.subs({x: 0}))

Delta = sp.expand(B**2 - 4*A*C)

# Expected discriminant form: 4*((1+p**2)*R**2 - (p*x0 - y0 + r)**2)
Delta_expected = sp.expand(4*((1+p**2)*R**2 - (p*x0 - y0 + r)**2))

print("Quadratic in x: A x^2 + B x + C = 0")
print("A =", A)
print("B =", B)
print("C =", C)

print("\nDiscriminant Δ = B^2 - 4 A C")
print("Δ  =", Delta)
print("Δ_expected =", Delta_expected)
print("Matches:", sp.simplify(Delta - Delta_expected) == 0)

# Tangency condition: Δ = 0  ⇔  |p x0 - y0 + r| / sqrt(1+p^2) = R
distance = sp.Abs(p*x0 - y0 + r) / sp.sqrt(1 + p**2)

print("\nTangency (Δ=0)  ⇔  (1+p^2) R^2 = (p x0 - y0 + r)^2  ⇔  distance = R")
print("distance(center, line) =", distance)

Quadratic in x: A x^2 + B x + C = 0
A = p**2 + 1
B = 2*p*r - 2*p*y0 - 2*x0
C = -R**2 + r**2 - 2*r*y0 + x0**2 + y0**2

Discriminant Δ = B^2 - 4 A C
Δ  = 4*R**2*p**2 + 4*R**2 - 4*p**2*x0**2 - 8*p*r*x0 + 8*p*x0*y0 - 4*r**2 + 8*r*y0 - 4*y0**2
Δ_expected = 4*R**2*p**2 + 4*R**2 - 4*p**2*x0**2 - 8*p*r*x0 + 8*p*x0*y0 - 4*r**2 + 8*r*y0 - 4*y0**2
Matches: True

Tangency (Δ=0)  ⇔  (1+p^2) R^2 = (p x0 - y0 + r)^2  ⇔  distance = R
distance(center, line) = Abs(p*x0 + r - y0)/sqrt(p**2 + 1)


# 8

In [ ]:
# SymPy program for Exercise 8: rotate to remove the mixed term and report the diagonalized coefficients.
# - Uses the identity tan(2θ) = 2b / (a - c), implemented via θ = 0.5 * atan2(2b, a - c).
# - Confirms the mixed term vanishes at θ*, and returns the diagonal entries (which equal the eigenvalues).
# - Provides a convenience function with optional numeric substitution.
#
# No plots, just symbolic computation.

import sympy as sp

# Symbols
a, b, c = sp.symbols('a b c', real=True)
u, v = sp.symbols('u v', real=True)
theta = sp.symbols('theta', real=True)

# Rotation matrix R(θ) taking (u,v) -> (x,y)
R = sp.Matrix([[sp.cos(theta), -sp.sin(theta)],
               [sp.sin(theta),  sp.cos(theta)]])

# Quadratic-form matrix for ax^2 + 2bxy + cy^2
A = sp.Matrix([[a, b],
               [b, c]])

# Rotated quadratic form in (u,v): [u v] * (R^T A R) * [u v]^T
B = sp.simplify(R.T * A * R)

# Extract coefficients so that ax^2+2bxy+cy^2 becomes α u^2 + 2 β u v + γ v^2
alpha_theta = sp.simplify(B[0,0])
beta_theta  = sp.simplify(B[0,1])  # this is "β"; mixed term is 2*β u v
gamma_theta = sp.simplify(B[1,1])

# Closed-form rotation that kills the mixed term: tan(2θ) = 2b/(a-c)  ⇒ θ* = 1/2 atan2(2b, a-c)
theta_star = sp.simplify(sp.atan2(2*b, a - c) / 2)

# Evaluate coefficients at θ*
alpha_star = sp.simplify(alpha_theta.subs(theta, theta_star))
beta_star  = sp.simplify(beta_theta.subs(theta, theta_star))
gamma_star = sp.simplify(gamma_theta.subs(theta, theta_star))

# Eigenvalues of A for reference (should match alpha_star and gamma_star up to ordering)
eigvals = [sp.simplify(ev) for ev in A.eigenvals().keys()]

# Package in a function
def diagonalize_quadratic(a_sym=a, b_sym=b, c_sym=c, subs=None):
    """
    Return a dict with: theta_star, alpha_star, beta_star (should be 0), gamma_star,
    and eigenvalues of A. If 'subs' is provided (dict mapping symbols to numbers),
    evaluate numerically as well.
    """
    res = {
        "theta_star": theta_star,
        "alpha_theta": alpha_theta,
        "beta_theta": beta_theta,
        "gamma_theta": gamma_theta,
        "alpha_star": alpha_star,
        "beta_star": beta_star,
        "gamma_star": gamma_star,
        "eigenvalues": eigvals,
    }
    if subs:
        res_num = {
            "theta_star_num": sp.N(theta_star.subs(subs)),
            "alpha_star_num": sp.N(alpha_star.subs(subs)),
            "beta_star_num": sp.N(beta_star.subs(subs)),
            "gamma_star_num": sp.N(gamma_star.subs(subs)),
            "eigenvalues_num": [sp.N(ev.subs(subs)) for ev in eigvals],
        }
        res.update(res_num)
    return res

# Demo 1 (symbolic): show the mixed term vanishes
print("β(θ*) =", beta_star)  # should simplify to 0

# Demo 2 (numeric): pick a,b,c and show diagonalized coefficients and angle
example = {a: 5, b: 3, c: 1}
res = diagonalize_quadratic(subs=example)

print("\nExample with a=5, b=3, c=1")
for k in ["theta_star_num", "alpha_star_num", "beta_star_num", "gamma_star_num", "eigenvalues_num"]:
    print(f"{k}: {res[k]}")

# Show that alpha_star/gamma_star match eigenvalues (order may differ)
print("\nSymbolic eigenvalues of A:")
for ev in eigvals:
    print("  ", ev)

print("\nDiagonalized quadratic at θ*:  alpha*u^2 + gamma*v^2 = 1  (mixed term removed)")

β(θ*) = 0

Example with a=5, b=3, c=1
theta_star_num: 0.491396861623665
alpha_star_num: 6.60555127546399
beta_star_num: 0
gamma_star_num: -0.605551275463989
eigenvalues_num: [-0.605551275463989, 6.60555127546399]

Symbolic eigenvalues of A:
   a/2 + c/2 - sqrt(a**2 - 2*a*c + 4*b**2 + c**2)/2
   a/2 + c/2 + sqrt(a**2 - 2*a*c + 4*b**2 + c**2)/2

Diagonalized quadratic at θ*:  alpha*u^2 + gamma*v^2 = 1  (mixed term removed)


# 9

In [18]:
# Exercise 9 — SymPy program (plain output, no special formatting)
# - Builds the 3x3 linear system for (m, n, q)
# - Proves det(C) = 8 * A_signed symbolically
# - Solves for (m, n, q), center, and R^2 for non-collinear examples
# - Detects/flags the collinear (singular) case
#
# Keep it simple: just print/pprint the results.

import sympy as sp

# Symbols
x1, y1, x2, y2, x3, y3 = sp.symbols('x1 y1 x2 y2 x3 y3', real=True)
m, n, q = sp.symbols('m n q', real=True)

# Matrices
C = sp.Matrix([
    [2*x1, 2*y1, 1],
    [2*x2, 2*y2, 1],
    [2*x3, 2*y3, 1],
])
rhs = -sp.Matrix([x1**2 + y1**2, x2**2 + y2**2, x3**2 + y3**2])

M = sp.Matrix([
    [x1, y1, 1],
    [x2, y2, 1],
    [x3, y3, 1],
])

# Determinant/area relation
detC = sp.simplify(C.det())
detM = sp.simplify(M.det())
A_signed = sp.simplify(detM / 2)  # oriented (signed) area

print("Linear system:  C * [m, n, q]^T = rhs")
sp.pprint(C)
sp.pprint(rhs)

print("\nDeterminant/area relation:")
print("det(C) =", detC)
print("det(M) =", detM)
print("A_signed = det(M)/2 =", A_signed)
print("Check det(C) - 8*A_signed =", sp.simplify(detC - 8*A_signed))

# Solver function
def solve_three_point_circle(P1, P2, P3, tol=sp.Float("1e-12")):
    """Given three points Pk=(xk,yk), return dict with solution or collinearity flag."""
    Xi = [sp.nsimplify(P1[0]), sp.nsimplify(P2[0]), sp.nsimplify(P3[0])]
    Yi = [sp.nsimplify(P1[1]), sp.nsimplify(P2[1]), sp.nsimplify(P3[1])]
    Cnum = sp.Matrix([
        [2*Xi[0], 2*Yi[0], 1],
        [2*Xi[1], 2*Yi[1], 1],
        [2*Xi[2], 2*Yi[2], 1],
    ])
    rhsnum = -sp.Matrix([Xi[0]**2 + Yi[0]**2, Xi[1]**2 + Yi[1]**2, Xi[2]**2 + Yi[2]**2])
    Mnum = sp.Matrix([
        [Xi[0], Yi[0], 1],
        [Xi[1], Yi[1], 1],
        [Xi[2], Yi[2], 1],
    ])
    dM = sp.N(Mnum.det())
    if abs(dM) < tol:
        return {"collinear": True, "detM": dM}
    sol = Cnum.LUsolve(rhsnum)
    mval, nval, qval = map(sp.simplify, sol)
    center = sp.Matrix([-mval, -nval])
    R2 = sp.simplify(mval**2 + nval**2 - qval)
    return {
        "collinear": False,
        "m": sp.N(mval), "n": sp.N(nval), "q": sp.N(qval),
        "center": sp.N(center),
        "R2": sp.N(R2),
        "detM": sp.N(dM),
        "A_signed": sp.N(dM/2),
    }

# Examples
print("\nExample (non-collinear): P1=(0,0), P2=(1,0), P3=(0,1)")
ex1 = solve_three_point_circle((0,0), (1,0), (0,1))
for k,v in ex1.items():
    print(f"{k}: {v}")

print("\nExample (collinear): P1=(0,0), P2=(1,1), P3=(2,2)")
ex2 = solve_three_point_circle((0,0), (1,1), (2,2))
for k,v in ex2.items():
    print(f"{k}: {v}")

Linear system:  C * [m, n, q]^T = rhs
⎡2⋅x₁  2⋅y₁  1⎤
⎢             ⎥
⎢2⋅x₂  2⋅y₂  1⎥
⎢             ⎥
⎣2⋅x₃  2⋅y₃  1⎦
⎡    2     2⎤
⎢- x₁  - y₁ ⎥
⎢           ⎥
⎢    2     2⎥
⎢- x₂  - y₂ ⎥
⎢           ⎥
⎢    2     2⎥
⎣- x₃  - y₃ ⎦

Determinant/area relation:
det(C) = 4*x1*y2 - 4*x1*y3 - 4*x2*y1 + 4*x2*y3 + 4*x3*y1 - 4*x3*y2
det(M) = x1*y2 - x1*y3 - x2*y1 + x2*y3 + x3*y1 - x3*y2
A_signed = det(M)/2 = x1*y2/2 - x1*y3/2 - x2*y1/2 + x2*y3/2 + x3*y1/2 - x3*y2/2
Check det(C) - 8*A_signed = 0

Example (non-collinear): P1=(0,0), P2=(1,0), P3=(0,1)
collinear: False
m: -0.500000000000000
n: -0.500000000000000
q: 0
center: Matrix([[0.500000000000000], [0.500000000000000]])
R2: 0.500000000000000
detM: 1.00000000000000
A_signed: 0.500000000000000

Example (collinear): P1=(0,0), P2=(1,1), P3=(2,2)
collinear: True
detM: 0


# 10

In [19]:
# Exercise 10 — SymPy program: classify each implicit curve; for circles, extract center & radius
# Plain printed output (no LaTeX/HTML).

import sympy as sp

x, y = sp.symbols('x y', real=True)

# Define the four equations' left-hand sides
exprs = {
    "(a)": x**2 + y**2 - 6*x + 8*y + 9,
    "(b)": 2*x**2 + 2*y**2 + 4*x - 12*y + 1,
    "(c)": x**2 + y**2 + 4*x + 6*y + 20,
    "(d)": x**2 + 2*x*y + y**2 - 16,
}

def circle_params_from_expr(expr):
    """
    Try to match expr to x^2 + y^2 + 2 m x + 2 n y + q.
    Return (is_isotropic, m, n, q) if matchable; else (False, None, None, None).
    """
    # Collect coefficients
    poly = sp.Poly(expr, x, y)
    a = poly.coeffs()[poly.monoms().index((2,0))] if (2,0) in poly.monoms() else 0
    c = poly.coeffs()[poly.monoms().index((0,2))] if (0,2) in poly.monoms() else 0
    b = poly.coeffs()[poly.monoms().index((1,1))] if (1,1) in poly.monoms() else 0
    dx = poly.coeffs()[poly.monoms().index((1,0))] if (1,0) in poly.monoms() else 0
    ey = poly.coeffs()[poly.monoms().index((0,1))] if (0,1) in poly.monoms() else 0
    f  = poly.coeffs()[poly.monoms().index((0,0))] if (0,0) in poly.monoms() else 0

    # Isotropic quadratic part?
    isotropic = (b == 0) and (a == c) and (a != 0)

    if isotropic:
        # normalize by 'a' to match x^2 + y^2 + 2mx + 2ny + q
        m = sp.Rational(dx, 2*a)
        n = sp.Rational(ey, 2*a)
        q = sp.Rational(f, a)
        return True, sp.simplify(m), sp.simplify(n), sp.simplify(q)
    else:
        return False, None, None, None

def classify(expr):
    # Try circle form
    is_circle_form, m, n, q = circle_params_from_expr(expr)
    if is_circle_form:
        S = sp.simplify(m**2 + n**2 - q)  # R^2
        if sp.simplify(S) > 0:
            return {
                "type": "circle",
                "center": (-m, -n),
                "R2": S,
                "radius": sp.sqrt(S),
                "degenerate": False
            }
        elif sp.simplify(S) == 0:
            return {
                "type": "point circle (degenerate)",
                "center": (-m, -n),
                "R2": S,
                "radius": 0,
                "degenerate": True
            }
        else:
            return {
                "type": "empty locus (imaginary circle)",
                "center": (-m, -n),
                "R2": S,
                "radius": sp.I*sp.sqrt(-S),  # purely formal
                "degenerate": True
            }
    else:
        # Not a circle (mixed term or anisotropic). Try factoring to see structure.
        fact = sp.factor(expr)
        return {
            "type": "not a circle (anisotropic or mixed term present)",
            "factorization": fact
        }

# Run classifications
for label, e in exprs.items():
    result = classify(e)
    print(f"{label}  {sp.simplify(e)} = 0")
    if result["type"] == "circle":
        print("  Classification: circle")
        print("  Center:", sp.simplify(result["center"]))
        print("  R^2:", sp.simplify(result["R2"]))
        print("  Radius:", sp.simplify(result["radius"]))
    elif result["type"] == "point circle (degenerate)":
        print("  Classification: point circle (degenerate)")
        print("  Center:", sp.simplify(result["center"]))
        print("  R^2:", sp.simplify(result["R2"]))
        print("  Radius:", 0)
    elif result["type"] == "empty locus (imaginary circle)":
        print("  Classification: empty locus (imaginary circle)")
        print("  Center:", sp.simplify(result["center"]))
        print("  R^2:", sp.simplify(result["R2"]))
    else:
        print("  Classification:", result["type"])
        print("  Factorization:", result["factorization"])
    print()

(a)  x**2 - 6*x + y**2 + 8*y + 9 = 0
  Classification: circle
  Center: (3, -4)
  R^2: 16
  Radius: 4

(b)  2*x**2 + 4*x + 2*y**2 - 12*y + 1 = 0
  Classification: circle
  Center: (-1, 3)
  R^2: 19/2
  Radius: sqrt(38)/2

(c)  x**2 + 4*x + y**2 + 6*y + 20 = 0
  Classification: empty locus (imaginary circle)
  Center: (-2, -3)
  R^2: -7

(d)  x**2 + 2*x*y + y**2 - 16 = 0
  Classification: not a circle (anisotropic or mixed term present)
  Factorization: (x + y - 4)*(x + y + 4)

